# PPO from frozen v39 BC
Loads an immutable BC artifact. This notebook never parses replays or retrains BC.

In [ ]:
SMOKE_MODE = False
ROUNDS = 2 if SMOKE_MODE else 10
EPISODES_PER_ROUND = 10 if SMOKE_MODE else 100
print(f'PPO-only: smoke={SMOKE_MODE} rounds={ROUNDS} episodes/round={EPISODES_PER_ROUND}')

In [ ]:
import glob, os, sys
for package in ('src', 'cg'):
    matches = glob.glob(f'/kaggle/input/**/{package}/__init__.py', recursive=True)
    if not matches:
        raise RuntimeError(f'Mounted package not found: {package}')
    package_root = os.path.dirname(os.path.dirname(matches[0]))
    sys.path.insert(0, package_root)
    print(f'{package} root: {package_root}')
config_matches = glob.glob('/kaggle/input/**/deck_lucario.csv', recursive=True)
if not config_matches:
    raise RuntimeError('Mounted deck configs not found')
CONFIG_DIR = os.path.dirname(config_matches[0])
bc_matches = glob.glob('/kaggle/input/**/weights.npz', recursive=True)
if len(bc_matches) != 1:
    raise RuntimeError(f'Expected exactly one v39 BC weights.npz, found: {bc_matches}')
BC_WEIGHTS = bc_matches[0]
print(f'Frozen BC artifact: {BC_WEIGHTS}')

In [ ]:
import copy, json, jax, jax.numpy as jnp, numpy as np
from cg.game import battle_start, battle_select, battle_finish
from src.featurizer import feature_dims
from src.policy_numpy import export_params, load_params
from src.ppo import PPOConfig, init_ppo_state
from src.selfplay import SelfPlayConfig, load_decks, run_selfplay
dims = feature_dims()
sample = {'board': np.zeros(dims['B'], np.float32), 'card_ids': np.zeros(dims['C'], np.int32), 'options': np.zeros((dims['K'], dims['O']), np.float32), 'option_card': np.zeros((dims['K'], dims['Q']), np.int32), 'legal_mask': np.zeros(dims['K'], np.float32), 'deck_id': np.int32(0)}
ppo_cfg = PPOConfig(lr=1e-5, embed_dim=64, trunk_hidden=512, clip_eps=0.2, entropy_coef=0.01, epochs_per_batch=4, minibatch_size=128, target_kl=0.02, kl_coef=0.2)
state = init_ppo_state(ppo_cfg, sample)
loaded = load_params(BC_WEIGHTS)
template = state.params['params'] if 'params' in state.params else state.params
def shapes(tree):
    return {tuple(path): tuple(np.asarray(value).shape) for path, value in jax.tree_util.tree_flatten_with_path(tree)[0]}
if shapes(loaded) != shapes(template):
    raise ValueError('v39 BC parameter names/shapes do not match the 64/512 PPO model')
params = {'params': loaded} if 'params' in state.params else loaded
state = state.replace(params=jax.tree_util.tree_map(jnp.asarray, params))
print(f'Loaded and validated {len(jax.tree_util.tree_leaves(params))} BC arrays')
class CgEngine:
    def battle_start(self, d0, d1):
        obs, _ = battle_start(d0, d1); return obs
    def battle_select(self, action): return battle_select(action)
    def battle_finish(self): battle_finish()
decks = load_decks(CONFIG_DIR, ['Lucario', 'Crustle', 'Alakazam'])
cfg = SelfPlayConfig(episodes_per_round=EPISODES_PER_ROUND, rounds=ROUNDS, deck_lists=decks, deck_names=list(decks), ppo=ppo_cfg, temperature=1.0, pool_size=4, n_parallel=4, frozen_opp_frac=0.4, screen_eval_games=10 if SMOKE_MODE else 20, final_eval_games=20 if SMOKE_MODE else 200, worker_candidates=(4, 8) if SMOKE_MODE else None, checkpoint_dir='/kaggle/working/weights/ppo')
summary = run_selfplay(CgEngine(), state, copy.deepcopy(state), cfg, ref_params=state.params)
os.makedirs('/kaggle/working/weights/ppo', exist_ok=True)
export_params(summary['state_a'].params, '/kaggle/working/weights/ppo/ppo_a.npz')
export_params(summary['state_b'].params, '/kaggle/working/weights/ppo/ppo_b.npz')
export_params(summary['best_params'], '/kaggle/working/weights/ppo/ppo_best.npz')
np.savez('/kaggle/working/weights/ppo/bc_source.npz', **dict(np.load(BC_WEIGHTS)))
metric_keys = ['best_bc_win_rate', 'final_best_bc_win_rate', 'final_best_bc_n', 'final_best_bc_wins', 'final_best_bc_losses', 'final_best_bc_draws', 'final_best_bc_errors', 'final_best_bc_elapsed_s', 'final_best_bc_games_per_s', 'worker_bench']
metrics = {key: summary.get(key) for key in metric_keys if key in summary}
with open('/kaggle/working/weights/ppo/metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
print(f'Final BC evaluation: {metrics}')
print(f"PPO complete: {summary['final_elo']}")